# ShellForge: private repository ZIP + resumable cloud GPU workflow
1. While signed in to GitHub, open the PR branch **codex/review2-cloud**, then choose **Code > Download ZIP**.
2. For a private repo, download this `.ipynb` from GitHub and open it in Colab via **File > Upload notebook**. Select **Runtime > Change runtime type > T4 GPU**.
3. Choose **Runtime > Run all**. Upload the ZIP when prompted and authorize Google Drive when prompted.
No GitHub token, git clone, or code editing is required. Keep the same ZIP for resuming a run.
After a disconnect: reconnect T4, Run all, upload the same ZIP, and authorize Drive again. Saved checkpoints and predictions are reused.
Training/results are genuine only after the run completes; providing this notebook does not claim improvement.


In [ ]:
"""Token-free ZIP bootstrap. Also embedded verbatim in the first notebook cell."""
import hashlib
import json
from pathlib import Path, PurePosixPath
import re
import shutil
import stat
import subprocess
import tempfile
import zipfile

SOURCE_ROOTS = ('src', 'scripts', 'benchmarks')
SOURCE_FILES = ('requirements.txt', 'requirements-model.txt', 'requirements-colab.txt',
                'artifacts/qwen-lora/training_manifest.json',
                'reports/metrics/base/generation.json',
                'reports/metrics/base/generation_examples.jsonl')


def source_fingerprint(root):
    root = Path(root)
    paths = {p for name in SOURCE_ROOTS for p in (root / name).rglob('*')
             if p.is_file() and '__pycache__' not in p.parts and p.suffix != '.pyc'}
    paths.update(root / name for name in SOURCE_FILES)
    paths.update((root / 'data/review2').glob('*.jsonl.gz'))
    hasher = hashlib.sha256()
    for path in sorted(paths):
        hasher.update(path.relative_to(root).as_posix().encode() + b'\0')
        hasher.update(hashlib.sha256(path.read_bytes()).digest())
    return hasher.hexdigest()


def source_provenance(root):
    root = Path(root)
    marker = root / 'source_provenance.json'
    if marker.exists():
        result = json.loads(marker.read_text())
        if result['content_sha256'] != source_fingerprint(root):
            raise ValueError('Uploaded runtime source changed since extraction')
        return result
    commit = None
    if (root / '.git').exists():
        commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip()
    return {'kind': 'git' if commit else 'directory', 'git_commit': commit,
            'content_sha256': source_fingerprint(root)}


def extract_repository_zip(archive_path, destination):
    archive_path, destination = Path(archive_path), Path(destination)
    archive_hash = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    with zipfile.ZipFile(archive_path) as archive, tempfile.TemporaryDirectory(dir=destination.parent) as temporary:
        names = set()
        total = 0
        for info in archive.infolist():
            path = PurePosixPath(info.filename)
            if (path.is_absolute() or '..' in path.parts or '\\' in info.filename or
                    ':' in info.filename or info.filename in names or
                    stat.S_ISLNK(info.external_attr >> 16) or '.git' in path.parts):
                raise ValueError('ZIP contains unsafe or duplicate paths; use GitHub Code > Download ZIP')
            names.add(info.filename)
            total += info.file_size
            if total > 2 * 1024**3:
                raise ValueError('ZIP expands beyond 2 GiB; upload the repository source ZIP only')
        archive.extractall(temporary)
        candidates = [p.parent.parent for p in Path(temporary).rglob('scripts/run_gpu_experiments.py')
                      if (p.parent.parent / 'src/train.py').is_file()]
        if len(candidates) != 1:
            raise ValueError('ZIP must contain exactly one ShellForge repository root')
        root = candidates[0]
        for name in (*SOURCE_FILES, 'src/run_state.py', 'src/archive_source.py',
                     'data/review2/train.jsonl.gz', 'data/review2/validation.jsonl.gz', 'data/review2/test.jsonl.gz'):
            if not (root / name).is_file():
                raise ValueError('ZIP is outdated or incomplete. Download the latest PR branch ZIP.')
        comment = archive.comment.decode('ascii', errors='replace').strip()
        provenance = {'kind': 'uploaded_zip', 'git_commit': None,
                      'archive_declared_commit': comment if re.fullmatch('[0-9a-fA-F]{40}', comment) else None,
                      'commit_verified': False, 'archive_sha256': archive_hash,
                      'content_sha256': source_fingerprint(root)}
        if destination.exists():
            if source_fingerprint(destination) != provenance['content_sha256']:
                raise ValueError('A different source is already extracted. Reconnect a fresh runtime and upload the same ZIP used by the run.')
            return destination, source_provenance(destination)
        shutil.copytree(root, destination)
        (destination / 'source_provenance.json').write_text(json.dumps(provenance, indent=2) + '\n')
        return destination, provenance

import os
import sys
from google.colab import files
print('Upload the latest ShellForge PR branch ZIP (GitHub: Code > Download ZIP). No token is needed.')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Please upload exactly one repository ZIP, then rerun this cell.')
filename, payload = next(iter(uploaded.items()))
if not filename.lower().endswith('.zip'):
    raise ValueError('Please select the repository .zip file.')
archive_path = Path('/content/ShellForge-upload.zip')
archive_path.write_bytes(payload)
repo, provenance = extract_repository_zip(archive_path, Path('/content/ShellForge'))
os.chdir(repo)
sys.path.insert(0, str(repo))
print('Repository:', repo)
print('ZIP/source provenance:', json.dumps(provenance, indent=2))


In [ ]:
# Isolate PyTorch from Colab's preinstalled torchvision/torchaudio packages.
environment = Path('/content/shellforge-env')
python = environment / 'bin/python'
if not python.exists():
    subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(environment)], check=True)
pip_command = [sys.executable, '-m', 'pip', '--python', str(python)]
subprocess.run(pip_command + ['install', 'torch==2.14.1', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run(pip_command + ['install', '-r', 'requirements-colab.txt'], check=True)
subprocess.run(['apt-get', 'update'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'poppler-utils', 'fonts-dejavu-core'], check=True)


In [ ]:
subprocess.run([str(python), "-c", "import torch; assert torch.cuda.is_available(), 'Connect a T4 GPU runtime'; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0)); print('GPU memory GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30); print('CUDA kernel check:', torch.ones(1, device='cuda').item())"], check=True)


## Persist evidence in cloud storage
Mount Google Drive so logs, optimizer checkpoints and the final adapter survive a Colab disconnect.
The base model downloads into Colab's ephemeral cache. Run output stays in Drive.
Run all automatically resumes the same Drive folder; completed trials/evaluations are reused, and mismatched source/data/settings are rejected.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
run_dir = Path('/content/drive/MyDrive/ShellForge/full-data-v1')


In [ ]:
subprocess.run([str(python), '-m', 'src.experiment_data', '--restore'], check=True)
test_log = repo / 'reports/logs/full_data_tests.log'
with test_log.open('w') as log:
    subprocess.run([str(python), '-m', 'pytest', '-q'], check=True, stdout=log, stderr=subprocess.STDOUT)
print(test_log.read_text())


## Bounded training and model selection
All quality-filtered training examples; all 1,255 validation examples for generation selection.
Proposed settings: 3 epochs maximum, rank 16, attention + MLP adapters, batch 2 x accumulation 8,
512-token supervised budget, cosine decay, 5% warmup, learning rates 1e-4 and 5e-5, seed 42.
FP16 on T4; BF16 only when supported. Ordinary LoRA fits this 0.5B model; QLoRA is optional.
Checkpoint every 100 steps; retain lowest validation-loss checkpoint, patience 3.
Compare candidates by validation strict exact match, tie-break on loss. Settings are frozen before test.
If a trial runs out of GPU memory, preserve its logs and reduce batch size while increasing accumulation
before any test has run. Do not edit settings based on test results.


In [ ]:
# Long-running cloud GPU job. Logs stream here and are saved under run_dir/logs.
subprocess.run([str(python), '-u', 'scripts/run_gpu_experiments.py',
                '--output', str(run_dir), '--resume'], check=True)


## Inspect genuine results
The final comparison uses exactly the same 1,254 examples and greedy decoding for base and tuned.
The primary analysis contains the 1,222 examples not exposed in the historical 32-case pilot;
full-test results are also saved and explicitly disclose the 32 previously exposed examples.
Includes strict match, Wilson intervals, paired McNemar test, syntax validity, conservative AST trivia
normalization, AST node-kind overlap, category metrics, and separate safety classification.
AST metrics do not prove semantic equivalence. Review semantic_review.jsonl before claiming equivalence.
No generated command is ever executed. If validation does not improve, no improvement is claimed.


In [ ]:
import json
completion = json.loads((run_dir / 'completion.json').read_text())
comparison = json.loads((run_dir / 'comparison.json').read_text())
print(json.dumps(completion, indent=2))
print(json.dumps(comparison['primary_unexposed_test'], indent=2))
selection = json.loads((run_dir / 'selection.json').read_text())
adapter = selection['selected_adapter']
print('Adapter:', adapter)
subprocess.run([str(python), '-m', 'src.inference', 'Show the current working directory',
                '--backend', 'hf', '--revision', selection['revision'], '--adapter', adapter], check=True)


## Regenerate the four-page university report from saved evidence
The report keeps all six required sections and the negative pilot results. The builder refuses to use
incomplete runs and adds only measured full-data results. Report generation remains in Colab.
To continue analysis in Codex, share the small comparison/manifest/log files or a cloud link;
keep the model weights in Drive.


In [ ]:
if not all((run_dir / name).exists() for name in ['ShellForge_Review2.pdf', 'ShellForge_Review2.md']):
    subprocess.run([str(python), 'scripts/build_report.py', '--experiment', str(run_dir)], check=True)
import shutil
for name in ['ShellForge_Review2.pdf', 'ShellForge_Review2.md']:
    if not (run_dir / name).exists():
        shutil.copy2(repo / 'reports' / name, run_dir / name)
subprocess.run([str(python), '-c', "from pypdf import PdfReader; import sys; p=PdfReader(sys.argv[1]); assert len(p.pages)==4; assert all(abs(float(x.mediabox.width)-595.276)<1 and abs(float(x.mediabox.height)-841.89)<1 for x in p.pages)", str(run_dir / 'ShellForge_Review2.pdf')], check=True)
if not (run_dir / 'review-1.png').exists():
    subprocess.run(['pdftoppm', '-scale-to', '1200', '-png', str(run_dir / 'ShellForge_Review2.pdf'), str(run_dir / 'review')], check=True)
from IPython.display import display, IFrame
from IPython.display import Image
for page in sorted(run_dir.glob('review-*.png')):
    display(Image(filename=str(page)))
print('Cloud report:', run_dir / 'ShellForge_Review2.pdf')
